## Imports and paths

In [31]:
import os
import sys
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

CLEANED_DATA_PATH = "../data/processed/hotel_bookings_cleaned.csv"

os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../reports", exist_ok=True)
os.makedirs("../models", exist_ok=True)

## Load cleaned dataset

In [32]:
df = pd.read_csv(CLEANED_DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (87138, 30)

Columns:
['hotel', 'is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 'booking_changes', 'deposit_type', 'agent', 'company', 'days_in_waiting_list', 'customer_type', 'adr', 'required_car_parking_spaces', 'total_of_special_requests']


## Verify the cleaned dataset

In [33]:
# Leakage columns must already be removed
assert "reservation_status" not in df.columns, \
    "ERROR: reservation_status is still present."

assert "reservation_status_date" not in df.columns, \
    "ERROR: reservation_status_date is still present."

# Target must exist
assert "is_canceled" in df.columns, \
    "ERROR: is_canceled target column is missing."

# Duplicates should already be removed
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

assert duplicate_count == 0, \
    "ERROR: Duplicate rows still exist in the cleaned dataset."

print("Dataset validation passed.")

Duplicate rows: 0
Dataset validation passed.


## Separate X and y

In [ ]:
X = df.drop(columns=["is_canceled"])
y = df["is_canceled"]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nTarget percentage:")
print(y.value_counts(normalize=True) * 100)


X shape: (87138, 29)
y shape: (87138,)

Target distribution:
is_canceled
0    63371
1    23767
Name: count, dtype: int64

Target percentage:
is_canceled
0    72.724873
1    27.275127
Name: proportion, dtype: float64


## Stratified Data Splitting

In [35]:
# First split:
# 85% = Train + Dev
# 15% = Test

X_train_dev, X_test, y_train_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    stratify=y,
    random_state=RANDOM_STATE
)

In [36]:
# Second split:
# 70% = Train
# 15% = Dev
#
# We already have 85% remaining.
# 15 / 85 = 0.17647

X_train, X_dev, y_train, y_dev = train_test_split(
    X_train_dev,
    y_train_dev,
    test_size=0.17647,
    stratify=y_train_dev,
    random_state=RANDOM_STATE
)

In [37]:
print("Train:", X_train.shape)
print("Dev:  ", X_dev.shape)
print("Test: ", X_test.shape)

print("\nTrain percentage:",
      len(X_train) / len(X) * 100)

print("Dev percentage:",
      len(X_dev) / len(X) * 100)

print("Test percentage:",
      len(X_test) / len(X) * 100)

Train: (60996, 29)
Dev:   (13071, 29)
Test:  (13071, 29)

Train percentage: 69.99931143703093
Dev percentage: 15.000344281484542
Test percentage: 15.000344281484542


## Stratification Validation

In [40]:
print("Original target distribution:")
print(y.value_counts(normalize=True) * 100)

print("\nTrain target distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\nDev target distribution:")
print(y_dev.value_counts(normalize=True) * 100)

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True) * 100)

Original target distribution:
is_canceled
0    72.724873
1    27.275127
Name: proportion, dtype: float64

Train target distribution:
is_canceled
0    72.724441
1    27.275559
Name: proportion, dtype: float64

Dev target distribution:
is_canceled
0    72.725882
1    27.274118
Name: proportion, dtype: float64

Test target distribution:
is_canceled
0    72.725882
1    27.274118
Name: proportion, dtype: float64


## Verify no row overlap

In [38]:
train_indices = set(X_train.index)
dev_indices = set(X_dev.index)
test_indices = set(X_test.index)

assert train_indices.isdisjoint(dev_indices)
assert train_indices.isdisjoint(test_indices)
assert dev_indices.isdisjoint(test_indices)

print("No row overlap between Train, Dev and Test.")

No row overlap between Train, Dev and Test.


## Save the split datasets

In [39]:
X_train.to_csv("../data/processed/X_train.csv", index=False)
X_dev.to_csv("../data/processed/X_dev.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)

y_train.to_csv("../data/processed/y_train.csv", index=False)
y_dev.to_csv("../data/processed/y_dev.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

print("Train, Dev and Test datasets saved successfully.")

Train, Dev and Test datasets saved successfully.
